In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# Helper to compute summary for a dataframe
def compact_summary(df, cmdb_col, name_col, name_limit=50):
    total_rows = int(df.shape[0])
    unique_cmdb = sorted(df[cmdb_col].dropna().unique().tolist())
    unique_names = sorted(df[name_col].dropna().unique().tolist())
    unique_names_limited = unique_names[:name_limit]
    min_ts = int(df['timestamp'].min()) if total_rows > 0 else None
    max_ts = int(df['timestamp'].max()) if total_rows > 0 else None
    if min_ts is not None:
        min_ts_utc8 = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        max_ts_utc8 = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    else:
        min_ts_utc8 = max_ts_utc8 = None
    # Compact one-row DataFrame summary
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'unique_cmdb_ids': unique_cmdb,
        f'unique_{name_col}_(first_{name_limit})': unique_names_limited,
        'min_timestamp_utc8': min_ts_utc8,
        'max_timestamp_utc8': max_ts_utc8
    }])
    return summary

metric_summary = compact_summary(metric_df, 'cmdb_id', 'kpi_name', name_limit=50)
trace_summary = compact_summary(trace_df, 'cmdb_id', 'trace_name', name_limit=50)

metric_summary, trace_summary

```
Out[1]:
```
```python
# Summarize previous results in plain English as a single compact string.
# Reuse existing metric_summary and trace_summary variables from earlier execution.

ms = metric_summary.iloc[0]
ts = trace_summary.iloc[0]

summary = (
    f"metric.csv: total_rows={int(ms['total_rows'])}; "
    f"unique_cmdb_ids={ms['unique_cmdb_ids']}; "
    f"unique_kpi_names_first_50={ms[f'unique_kpi_name_(first_50)']}; "
    f"min_timestamp_utc8={ms['min_timestamp_utc8']}; max_timestamp_utc8={ms['max_timestamp_utc8']}. "
    f"trace.csv: total_rows={int(ts['total_rows'])}; "
    f"unique_cmdb_ids={ts['unique_cmdb_ids']}; "
    f"unique_trace_names_first_50={ts[f'unique_trace_name_(first_50)']}; "
    f"min_timestamp_utc8={ts['min_timestamp_utc8']}; max_timestamp_utc8={ts['max_timestamp_utc8']}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                                    unique_cmdb_ids                         unique_kpi_name_(first_50)        min_timestamp_utc8        max_timestamp_utc8
0       91009  [db_001, db_002, db_003, db_004, db_005, db_00...  [ACS, AIOS, AWS, Agent_ping, Asm_Free_Tb, Buff...  2020-05-30 03:30:00+0800  2020-05-30 04:30:00+0800,    total_rows                                    unique_cmdb_ids                       unique_trace_name_(first_50)        min_timestamp_utc8        max_timestamp_utc8
0        6240  [docker_001, docker_002, docker_003, docker_00...  [trace.from_docker_001.remoteprocess.duration_...  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800)```
```

In [ ]:
# In[2]:

# Reuse existing metric_df loaded earlier. Compute global thresholds per (cmdb_id, kpi_name).
import pandas as pd
import numpy as np

# Ensure 'value' is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']

agg_df = grp.agg(
    count=lambda x: int(x.dropna().shape[0]),
    min=lambda x: float(x.min()) if x.dropna().shape[0] > 0 else np.nan,
    max=lambda x: float(x.max()) if x.dropna().shape[0] > 0 else np.nan,
    p95=lambda x: float(x.quantile(0.95)) if x.dropna().shape[0] > 0 else np.nan,
    p5=lambda x: float(x.quantile(0.05)) if x.dropna().shape[0] > 0 else np.nan
).reset_index()

# Sort and store in the required kernel variable name
thresholds_df = agg_df.sort_values(['cmdb_id', 'kpi_name']).reset_index(drop=True)

# Display a compact table: first 20 rows with requested columns
thresholds_df.head(20)

```
Out[2]:
```
```python
# Summarize the computed global thresholds as a plain English string.
# Reuse existing thresholds_df from the kernel.

n_pairs = int(thresholds_df.shape[0])
examples_df = thresholds_df.head(3)

example_strs = []
for _, r in examples_df.iterrows():
    example_strs.append(
        f"{r['cmdb_id']}:{r['kpi_name']} (count={int(r['count'])}, min={r['min']:.6g}, "
        f"max={r['max']:.6g}, p95={r['p95']:.6g}, p5={r['p5']:.6g})"
    )

summary = (
    f"Computed global thresholds for every (cmdb_id, kpi_name) pair across the entire metric.csv. "
    f"Total unique pairs = {n_pairs}. Examples: " + "; ".join(example_strs) +
    ". Results are stored in the kernel variable 'thresholds_df', sorted by cmdb_id then kpi_name."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name  count         min          max          p95          p5
0   db_001                  ACS     60    0.000000     0.030000     0.020000    0.010000
1   db_001                 AIOS     60    0.000000     0.020000     0.010000    0.000000
2   db_001                  AWS     60    0.000000     0.020000     0.020000    0.000000
3   db_001          Asm_Free_Tb     12   12.510000    12.510000    12.510000   12.510000
4   db_001         CPU_Used_Pct     49    0.882055     2.294741     2.250949    0.974492
5   db_001         CPU_free_pct     61   97.705259    99.117945    99.006543   97.769605
6   db_001         Call_Per_Sec     60   11.100000    18.300000    11.900000   11.290000
7   db_001  DFParaWrite_Per_Sec     60    0.000000     5.610000     3.091500    0.000000
8   db_001               DbTime     60    4.900000    74.500000    72.905000    4.900000
9   db_001         Exec_Per_Sec     60    9.900000   353.300000   320.465000   10.095000
10  db_001                 Hang     60    1.000000     1.000000     1.000000    1.000000
11  db_001  LFParaWrite_Per_Sec     60    0.000000    16.130000     2.892000    0.000000
12  db_001       LFSync_Per_Sec     60    0.000000     3.940000     2.747000    0.000000
13  db_001   Logic_Read_Per_Sec     60    3.900000  9600.100000  2501.800000    4.400000
14  db_001        Login_Per_Sec     60    0.100000     1.600000     0.300000    0.100000
15  db_001            MEM_Total     59  381.902264   381.902264   381.902264  381.902264
16  db_001             MEM_Used     59  245.763857   245.842564   245.823404  245.767835
17  db_001         MEM_Used_Pct     59   64.351985    64.498444    64.367968   64.352925
18  db_001        MEM_real_util     60   46.210000    46.370000    46.360000   46.210000
19  db_001      New_Tbs_Free_Gb      6  283.147441   283.147441   283.147441  283.147441```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df and thresholds_df from kernel
# Convert provided window bounds (UTC+8) to epoch seconds (UTC) for comparison with metric timestamps
start_dt = pd.to_datetime('2020-05-30 04:00:00+0800').tz_convert('UTC')
end_dt = pd.to_datetime('2020-05-30 04:30:00+0800').tz_convert('UTC')
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Filter metric_df to the incident window inclusive
metric_window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Join each metric row with its thresholds on (cmdb_id, kpi_name)
# thresholds_df has columns: cmdb_id, kpi_name, count, min, max, p95, p5
incident_window_df = metric_window_df.merge(
    thresholds_df,
    on=['cmdb_id', 'kpi_name'],
    how='inner',
    suffixes=('', '_th')
).copy()

# Ensure numeric
incident_window_df['value'] = pd.to_numeric(incident_window_df['value'], errors='coerce')
incident_window_df['p95'] = pd.to_numeric(incident_window_df['p95'], errors='coerce')
incident_window_df['p5'] = pd.to_numeric(incident_window_df['p5'], errors='coerce')

# Mark anomalies: value >= p95 OR value <= p5
incident_window_df['is_anomaly'] = (incident_window_df['value'] >= incident_window_df['p95']) | (incident_window_df['value'] <= incident_window_df['p5'])

# Aggregate per (cmdb_id, kpi_name) that has at least one anomaly
anoms = incident_window_df[incident_window_df['is_anomaly']].copy()

if anoms.shape[0] > 0:
    grp = anoms.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    summary_df = grp.agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        p95=('p95', 'first'),
        p5=('p5', 'first'),
        max_value_in_window=('value', 'max'),
        min_value_in_window=('value', 'min'),
        # keep a helper to decide type
        any_high=('value', lambda s: (s >= anoms.loc[s.index, 'p95']).any()),
    )

    # Determine anomaly_type and severity_ratio
    def compute_type_and_severity(row):
        any_high = bool(row['any_high'])
        if any_high:
            typ = 'high'
            denom = row['p95'] if (row['p95'] is not None and row['p95'] != 0) else 1e-12
            severity = row['max_value_in_window'] / denom
        else:
            typ = 'low'
            denom = max(row['min_value_in_window'], 1e-12)
            severity = row['p5'] / denom
        return pd.Series([typ, float(severity)])
    summary_df[['anomaly_type', 'severity_ratio']] = summary_df.apply(compute_type_and_severity, axis=1)

    # Convert earliest timestamp to UTC+8 string
    summary_df['earliest_anomaly_timestamp_utc8'] = summary_df['earliest_anomaly_timestamp'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )

    # Select and order requested columns
    result_df = summary_df[[
        'cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp_utc8',
        'anomaly_type', 'p95', 'p5', 'max_value_in_window', 'min_value_in_window', 'severity_ratio'
    ]].copy()

    # Sort by anomaly_count desc then severity_ratio desc
    result_df = result_df.sort_values(['anomaly_count', 'severity_ratio'], ascending=[False, False]).reset_index(drop=True)
else:
    # No anomalies found in window: return empty dataframe with requested columns
    result_df = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp_utc8',
        'anomaly_type', 'p95', 'p5', 'max_value_in_window', 'min_value_in_window', 'severity_ratio'
    ])

# Store full filtered-window-joined table in kernel variable incident_window_df (already set)
# Display compact top results (at most 20 rows)
result_df.head(20)

```
Out[3]:
```
```python
# Summarize the anomaly aggregation result in plain English as a compact string.
# Reuse result_df from the kernel.

df = result_df  # result_df created earlier
total_pairs = int(df.shape[0])

top_n = df.head(10)
entry_strs = []
for _, r in top_n.iterrows():
    entry_strs.append(
        f"{r['cmdb_id']}|{r['kpi_name']} (count={int(r['anomaly_count'])}, type={r['anomaly_type']}, "
        f"severity={float(r['severity_ratio']):.3g}, earliest={r['earliest_anomaly_timestamp_utc8']})"
    )

if total_pairs == 0:
    summary = "No anomalies found in the incident window 2020-05-30 04:00–04:30+0800."
else:
    summary = (
        f"Found {total_pairs} (cmdb_id, kpi_name) pairs with ≥1 anomaly in 2020-05-30 04:00–04:30+0800. "
        f"Top {min(total_pairs,10)} by anomaly_count / severity: " + "; ".join(entry_strs)
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name  anomaly_count earliest_anomaly_timestamp_utc8 anomaly_type        p95     p5  max_value_in_window  min_value_in_window  severity_ratio
0      db_007              MEM_real_util             31        2020-05-30 04:00:00+0800         high  83.520000  83.51            83.520000                83.52    1.000000e+00
1      os_007                Disk_rd_ios             31        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.000000                 0.00    0.000000e+00
2      os_020               Disk_io_util             31        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.000000                 0.00    0.000000e+00
3   redis_008  instantaneous_output_kbps             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             4.070000                 0.00    4.070000e+12
4      os_007                 Disk_svctm             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             2.300000                 0.00    2.300000e+12
5      os_021                 Disk_svctm             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             2.000000                 0.00    2.000000e+12
6      os_018                Disk_rd_ios             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             1.000000                 0.00    1.000000e+12
7   redis_008  instantaneous_ops_per_sec             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             1.000000                 0.00    1.000000e+12
8      os_019                    Page_pi             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.427609                 0.00    4.276090e+11
9      os_021               Disk_io_util             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.250000                 0.00    2.500000e+11
10     db_011                User_Commit             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.190000                 0.00    1.900000e+11
11     os_019               Disk_io_util             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.005546                 0.00    5.546333e+09
12     os_019                Disk_rd_kbs             30        2020-05-30 04:00:00+0800         high   0.000000   0.00             0.000259                 0.00    2.590000e+08
13     os_018                    Page_pi             30        2020-05-30 04:00:00+0800         high   0.066663   0.00           574.935301                 0.00    8.624549e+03
14     os_009                 Disk_await             30        2020-05-30 04:00:00+0800         high   0.080000   0.00             8.000000                 0.00    1.000000e+02
15  redis_005  instantaneous_output_kbps             30        2020-05-30 04:00:00+0800         high   0.060000   0.03             5.930000                 0.03    9.883333e+01
16  redis_008   instantaneous_input_kbps             30        2020-05-30 04:00:00+0800         high   0.000500   0.00             0.030000                 0.00    6.000000e+01
17     db_011                TPS_Per_Sec             30        2020-05-30 04:00:00+0800         high   0.005000   0.00             0.200000                 0.00    4.000000e+01
18     db_004              Login_Per_Sec             30        2020-05-30 04:00:00+0800         high   1.300000   1.20            50.100000                 1.20    3.853846e+01
19  redis_004   instantaneous_input_kbps             30        2020-05-30 04:00:00+0800         high   0.001500   0.00             0.040000                 0.00    2.666667e+01```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse incident_window_df from kernel; ensure timezone
tz = pytz.timezone('Asia/Shanghai')

# Work on anomaly points only
anom_points = incident_window_df[incident_window_df['is_anomaly']].copy()

# If no anomaly points, create empty faults_df and display
if anom_points.shape[0] == 0:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
        'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
        'min_value_in_fault','severity_ratio'
    ])
    top_faults = faults_df.head(20)
    top_faults
else:
    # Ensure sorted by cmdb_id,kpi_name,timestamp
    anom_points = anom_points.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # For each series, identify consecutive segments where gap <= 60 seconds
    def assign_segment(df):
        # df assumed sorted by timestamp
        df = df.copy()
        df['ts_diff'] = df['timestamp'].diff().fillna(0)
        # start of segment when ts_diff > 60
        df['new_seg'] = (df['ts_diff'] > 60).astype(int)
        df['seg_id'] = df['new_seg'].cumsum()
        return df

    anom_with_seg = anom_points.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_segment).reset_index(drop=True)

    # Aggregate per segment
    agg = anom_with_seg.groupby(['cmdb_id','kpi_name','seg_id'], as_index=False).agg(
        fault_start_ts=('timestamp','min'),
        fault_end_ts=('timestamp','max'),
        fault_length_minutes=('timestamp','count'),  # number of points
        any_high=('value', lambda s: (s >= anom_with_seg.loc[s.index, 'p95']).any()),
        p95=('p95','first'),
        p5=('p5','first'),
        max_value_in_fault=('value','max'),
        min_value_in_fault=('value','min')
    )

    # Determine anomaly_type and severity_ratio
    eps = 1e-12
    def compute_type_sev(row):
        if row['any_high']:
            typ = 'high'
            denom = row['p95'] if (row['p95'] is not None and row['p95'] != 0) else eps
            severity = row['max_value_in_fault'] / max(denom, eps)
        else:
            typ = 'low'
            denom = max(row['min_value_in_fault'], eps)
            severity = row['p5'] / denom
        return pd.Series([typ, float(severity)])
    agg[['anomaly_type','severity_ratio']] = agg.apply(compute_type_sev, axis=1)

    # Convert start/end timestamps to UTC+8 strings
    agg['fault_start_timestamp_utc8'] = agg['fault_start_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    agg['fault_end_timestamp_utc8'] = agg['fault_end_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )

    # Select required columns and filter severity_ratio >= 1.5
    faults_df = agg[[
        'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
        'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
        'min_value_in_fault','severity_ratio'
    ]].copy()

    faults_df = faults_df[faults_df['severity_ratio'] >= 1.5].reset_index(drop=True)

    # Sort by fault_length_minutes desc then severity_ratio desc
    faults_df = faults_df.sort_values(['fault_length_minutes','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

    # Provide compact top 20
    top_faults = faults_df.head(20)

    top_faults

```
Out[4]:
```
```python
# Summarize the fault segments result in plain English as a single compact string.
# Reuse faults_df from the kernel if available.

try:
    df = faults_df
except NameError:
    summary = "faults_df is not available in the kernel. The previous step may not have produced faults_df."
else:
    if df is None or df.shape[0] == 0:
        summary = "No fault segments with severity_ratio >= 1.5 were detected in the incident window 2020-05-30 04:00–04:30+0800."
    else:
        total_faults = int(df.shape[0])
        top = df.head(10)
        parts = []
        for _, r in top.iterrows():
            parts.append(
                f"{r['cmdb_id']}|{r['kpi_name']} start={r['fault_start_timestamp_utc8']} end={r['fault_end_timestamp_utc8']} "
                f"len={int(r['fault_length_minutes'])}min type={r['anomaly_type']} sev={float(r['severity_ratio']):.3g}"
            )
        summary = (
            f"Detected {total_faults} fault segment(s) (severity_ratio>=1.5) in 2020-05-30 04:00–04:30+08. "
            f"Top {min(total_faults,10)}: " + "; ".join(parts)
        )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
eps = 1e-12

# Check existence
if 'incident_window_df' not in globals():
    incident_window_df_missing = 'incident_window_df missing'
    incident_window_df_missing
else:
    # Reuse incident_window_df
    df = incident_window_df.copy()

    # Ensure numeric types
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['p95'] = pd.to_numeric(df['p95'], errors='coerce')
    df['p5'] = pd.to_numeric(df['p5'], errors='coerce')
    df['timestamp'] = df['timestamp'].astype(int)

    # 1) Ensure is_anomaly flag
    if 'is_anomaly' not in df.columns:
        df['is_anomaly'] = (df['value'] >= df['p95']) | (df['value'] <= df['p5'])
    else:
        # re-evaluate to be safe
        df['is_anomaly'] = (df['value'] >= df['p95']) | (df['value'] <= df['p5'])

    # Work only on anomaly points
    anoms = df[df['is_anomaly']].copy()
    if anoms.shape[0] == 0:
        faults_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
            'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
            'min_value_in_fault','severity_ratio'
        ])
        no_significant = pd.DataFrame({'message': ['no significant faults detected']})
        no_significant
    else:
        # Sort and assign segment ids per (cmdb_id,kpi_name) where gap > 60s starts a new segment
        anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
        # helper to assign seg idx
        def assign_seg(g):
            g = g.copy()
            g['ts_diff'] = g['timestamp'].diff().fillna(0)
            g['seg_index'] = (g['ts_diff'] > 60).cumsum()
            return g
        anoms = anoms.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_seg).reset_index(drop=True)

        # mark high points
        anoms['is_high_point'] = anoms['value'] >= anoms['p95']

        # Aggregate per segment
        agg = anoms.groupby(['cmdb_id','kpi_name','seg_index'], as_index=False).agg(
            fault_start_ts=('timestamp','min'),
            fault_end_ts=('timestamp','max'),
            fault_length_minutes=('timestamp','count'),
            any_high=('is_high_point', 'any'),
            p95=('p95','first'),
            p5=('p5','first'),
            max_value_in_fault=('value','max'),
            min_value_in_fault=('value','min')
        )

        # Determine anomaly_type and severity_ratio
        def compute_severity(row):
            if row['any_high']:
                typ = 'high'
                denom = row['p95'] if (row['p95'] is not None and row['p95'] != 0) else eps
                severity = row['max_value_in_fault'] / max(denom, eps)
            else:
                typ = 'low'
                denom = max(row['min_value_in_fault'], eps)
                severity = row['p5'] / denom
            return pd.Series([typ, float(severity)])
        agg[['anomaly_type','severity_ratio']] = agg.apply(compute_severity, axis=1)

        # Convert timestamps to UTC+8 strings
        agg['fault_start_timestamp_utc8'] = agg['fault_start_ts'].apply(
            lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        )
        agg['fault_end_timestamp_utc8'] = agg['fault_end_ts'].apply(
            lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        )

        # Select required columns
        faults_full = agg[[
            'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
            'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
            'min_value_in_fault','severity_ratio'
        ]].copy()

        # 4) Filter severity_ratio >= 1.5
        faults_df = faults_full[faults_full['severity_ratio'] >= 1.5].reset_index(drop=True)

        # Store faults_df in kernel (already assigned)
        # 5) Prepare compact table (top 20) sorted by fault_length_minutes desc then severity_ratio desc
        if faults_df.shape[0] == 0:
            no_significant = pd.DataFrame({'message': ['no significant faults detected']})
            no_significant
        else:
            faults_df = faults_df.sort_values(['fault_length_minutes','severity_ratio'], ascending=[False, False]).reset_index(drop=True)
            top_faults = faults_df.head(20)
            top_faults

```
Out[5]:
```
# Summarize the faults_df result in plain English as a single compact string.
# Reuse faults_df from the kernel if available.

try:
    df = faults_df
except NameError:
    summary = "faults_df missing"
else:
    if df is None or df.shape[0] == 0:
        summary = "No significant faults detected in the incident window 2020-05-30 04:00–04:30+0800."
    else:
        total = int(df.shape[0])
        top = df.head(10)
        parts = []
        for _, r in top.iterrows():
            parts.append(
                f"{r['cmdb_id']}|{r['kpi_name']} start={r['fault_start_timestamp_utc8']} "
                f"end={r['fault_end_timestamp_utc8']} len={int(r['fault_length_minutes'])}min "
                f"type={r['anomaly_type']} sev={float(r['severity_ratio']):.3g}"
            )
        summary = (
            f"Detected {total} significant fault segment(s) (severity_ratio>=1.5) in 2020-05-30 04:00–04:30+08. "
            f"Top {min(total,10)}: " + "; ".join(parts)
        )

summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
eps = 1e-12

# Ensure metric_df and thresholds_df exist
if 'metric_df' not in globals():
    raise RuntimeError("metric_df missing")
if 'thresholds_df' not in globals():
    raise RuntimeError("thresholds_df missing")

# Ensure incident_window_df exists; if not, build it
if 'incident_window_df' not in globals():
    # Convert window bounds (given in UTC+8) to UTC epoch seconds
    start_ts = int(pd.to_datetime('2020-05-30 04:00:00+0800').tz_convert('UTC').timestamp())
    end_ts = int(pd.to_datetime('2020-05-30 04:30:00+0800').tz_convert('UTC').timestamp())

    # Filter metric_df within window (timestamps are epoch seconds)
    incident_window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

    # Join with thresholds_df to attach p95 and p5
    incident_window_df = incident_window_df.merge(
        thresholds_df[['cmdb_id','kpi_name','p95','p5']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )

# Ensure numeric types
incident_window_df['value'] = pd.to_numeric(incident_window_df['value'], errors='coerce')
incident_window_df['p95'] = pd.to_numeric(incident_window_df['p95'], errors='coerce')
incident_window_df['p5'] = pd.to_numeric(incident_window_df['p5'], errors='coerce')
incident_window_df['timestamp'] = incident_window_df['timestamp'].astype(int)

# 1) add is_anomaly flag
incident_window_df['is_anomaly'] = (incident_window_df['value'] >= incident_window_df['p95']) | (incident_window_df['value'] <= incident_window_df['p5'])

# 2) detect segments among anomaly points
anoms = incident_window_df[incident_window_df['is_anomaly']].copy()
if anoms.shape[0] == 0:
    faults_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
        'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
        'min_value_in_fault','severity_ratio'
    ])
    output = pd.DataFrame({'message': ['no significant faults detected']})
else:
    anoms = anoms.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
    # mark high points
    anoms['is_high_point'] = anoms['value'] >= anoms['p95']

    # assign segment ids per (cmdb_id,kpi_name): new segment when gap > 60
    def assign_seg(g):
        g = g.copy()
        g['ts_diff'] = g['timestamp'].diff().fillna(0)
        g['seg_index'] = (g['ts_diff'] > 60).cumsum()
        return g

    anoms = anoms.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(assign_seg).reset_index(drop=True)

    # Aggregate per segment
    agg = anoms.groupby(['cmdb_id','kpi_name','seg_index'], as_index=False).agg(
        fault_start_ts=('timestamp','min'),
        fault_end_ts=('timestamp','max'),
        fault_length_minutes=('timestamp','count'),
        any_high=('is_high_point','any'),
        p95=('p95','first'),
        p5=('p5','first'),
        max_value_in_fault=('value','max'),
        min_value_in_fault=('value','min')
    )

    # determine anomaly_type and severity_ratio
    def compute_row_sev(r):
        if r['any_high']:
            typ = 'high'
            denom = r['p95'] if (r['p95'] is not None and r['p95'] != 0) else eps
            sev = r['max_value_in_fault'] / max(denom, eps)
        else:
            typ = 'low'
            denom = max(r['min_value_in_fault'], eps)
            sev = r['p5'] / denom
        return pd.Series([typ, float(sev)])
    agg[['anomaly_type','severity_ratio']] = agg.apply(compute_row_sev, axis=1)

    # convert timestamps to UTC+8 strings
    agg['fault_start_timestamp_utc8'] = agg['fault_start_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    agg['fault_end_timestamp_utc8'] = agg['fault_end_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )

    faults_full = agg[[
        'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_end_timestamp_utc8',
        'fault_length_minutes','anomaly_type','p95','p5','max_value_in_fault',
        'min_value_in_fault','severity_ratio'
    ]].copy()

    # 4) filter severity_ratio >= 1.5
    faults_df = faults_full[faults_full['severity_ratio'] >= 1.5].reset_index(drop=True)

    # store faults_df in kernel (already assigned)
    if faults_df.shape[0] == 0:
        output = pd.DataFrame({'message': ['no significant faults detected']})
    else:
        faults_df = faults_df.sort_values(['fault_length_minutes','severity_ratio'], ascending=[False, False]).reset_index(drop=True)
        output = faults_df.head(20)

# Return compact result in variable 'output' (and keep faults_df stored)
output

```
Out[6]:
```
```python
# Summarize faults_df result in plain English (compact).
try:
    df = faults_df
except NameError:
    summary = "faults_df missing"
else:
    if df is None or df.shape[0] == 0:
        summary = "no significant faults detected"
    else:
        total = int(df.shape[0])
        top = df.head(10)
        parts = []
        for _, r in top.iterrows():
            parts.append(
                f"{r['cmdb_id']}|{r['kpi_name']} start={r['fault_start_timestamp_utc8']} "
                f"end={r['fault_end_timestamp_utc8']} len={int(r['fault_length_minutes'])}min "
                f"type={r['anomaly_type']} sev={float(r['severity_ratio']):.3g}"
            )
        summary = f"Detected {total} significant fault segment(s). Top {min(total,10)}: " + "; ".join(parts)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name fault_start_timestamp_utc8  fault_end_timestamp_utc8  fault_length_minutes anomaly_type       p95    p5  max_value_in_fault  min_value_in_fault  severity_ratio
0   redis_008  instantaneous_output_kbps   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            4.070000                0.00    4.070000e+12
1      os_007                 Disk_svctm   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            2.300000                0.00    2.300000e+12
2      os_021                 Disk_svctm   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            2.000000                0.00    2.000000e+12
3      os_018                Disk_rd_ios   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            1.000000                0.00    1.000000e+12
4   redis_008  instantaneous_ops_per_sec   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            1.000000                0.00    1.000000e+12
5      os_019                    Page_pi   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            0.427609                0.00    4.276090e+11
6      os_021               Disk_io_util   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            0.250000                0.00    2.500000e+11
7      db_011                User_Commit   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            0.190000                0.00    1.900000e+11
8      os_019               Disk_io_util   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            0.005546                0.00    5.546333e+09
9      os_019                Disk_rd_kbs   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000000  0.00            0.000259                0.00    2.590000e+08
10     os_018                    Page_pi   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.066663  0.00          574.935301                0.00    8.624549e+03
11     os_009                 Disk_await   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.080000  0.00            8.000000                0.00    1.000000e+02
12  redis_005  instantaneous_output_kbps   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.060000  0.03            5.930000                0.03    9.883333e+01
13  redis_008   instantaneous_input_kbps   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000500  0.00            0.030000                0.00    6.000000e+01
14     db_011                TPS_Per_Sec   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.005000  0.00            0.200000                0.00    4.000000e+01
15     db_004              Login_Per_Sec   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  1.300000  1.20           50.100000                1.20    3.853846e+01
16  redis_004   instantaneous_input_kbps   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.001500  0.00            0.040000                0.00    2.666667e+01
17     db_006                       AIOS   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.000500  0.00            0.010000                0.00    2.000000e+01
18     os_007                 Disk_await   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  0.050000  0.00            1.000000                0.00    2.000000e+01
19     os_020                 Disk_svctm   2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800                    30         high  2.646500  0.00           35.520000                0.00    1.342150e+01```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Check faults_df existence
if 'faults_df' not in globals():
    faults_df_missing = 'faults_df missing'
    faults_df_missing
else:
    df = faults_df.copy()
    if df.shape[0] == 0:
        no_significant = pd.DataFrame({'message': ['no significant faults detected']})
        no_significant
    else:
        # 1) total number of fault segments
        total_fault_segments = int(df.shape[0])

        # 2) counts grouped by component prefix (token before first '_')
        df['component_prefix'] = df['cmdb_id'].astype(str).str.split('_', n=1).str[0]
        prefix_counts = df.groupby('component_prefix', as_index=False).size().rename(columns={'size':'count'})
        prefix_counts = prefix_counts.sort_values('count', ascending=False).reset_index(drop=True)

        # 3) overall earliest fault_start_timestamp_utc8 across all faults
        # Parse the existing UTC+8 timestamp strings into timezone-aware datetimes and find min
        parsed_starts = pd.to_datetime(df['fault_start_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        earliest_fault_start = parsed_starts.min()
        earliest_fault_start_utc8 = earliest_fault_start.strftime('%Y-%m-%d %H:%M:%S%z')

        # 4) top-10 table sorted by severity_ratio desc then fault_length_minutes desc
        df['severity_ratio'] = pd.to_numeric(df['severity_ratio'], errors='coerce')
        df['fault_length_minutes'] = pd.to_numeric(df['fault_length_minutes'], errors='coerce').astype(int)
        top_table = df.sort_values(['severity_ratio','fault_length_minutes'], ascending=[False, False]).head(10)
        top_table = top_table[[
            'cmdb_id','kpi_name','fault_start_timestamp_utc8','fault_length_minutes',
            'anomaly_type','p95','p5','max_value_in_fault','min_value_in_fault','severity_ratio'
        ]].reset_index(drop=True)

        # Display compact aggregates
        total_fault_segments, prefix_counts, earliest_fault_start_utc8, top_table

```
Out[7]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Check existence
if 'faults_df' not in globals():
    faults_df_missing = 'faults_df missing'
    faults_df_missing
else:
    df = faults_df.copy()
    # If empty
    if df.shape[0] == 0:
        summary = "no significant faults detected"
    else:
        # Total number of fault segments
        total_fault_segments = int(df.shape[0])

        # Counts grouped by component prefix (token before first '_')
        df['component_prefix'] = df['cmdb_id'].astype(str).str.split('_', n=1).str[0]
        prefix_counts_df = df.groupby('component_prefix', as_index=False).size().rename(columns={'size':'count'})
        prefix_counts_df = prefix_counts_df.sort_values('count', ascending=False).reset_index(drop=True)
        prefix_counts_list = [f"{row['component_prefix']}={int(row['count'])}" for _, row in prefix_counts_df.iterrows()]

        # Overall earliest fault_start_timestamp_utc8 across all faults
        # Parse timestamps (they include +0800) and get min, formatted as UTC+8 string
        parsed = pd.to_datetime(df['fault_start_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        earliest_fault_start_utc8 = parsed.min().strftime('%Y-%m-%d %H:%M:%S%z')

        # Top-10 table sorted by severity_ratio desc then fault_length_minutes desc
        df['severity_ratio'] = pd.to_numeric(df['severity_ratio'], errors='coerce')
        df['fault_length_minutes'] = pd.to_numeric(df['fault_length_minutes'], errors='coerce').astype(int)
        top_table = df.sort_values(['severity_ratio','fault_length_minutes'], ascending=[False, False]).head(10)
        top_entries = []
        for _, r in top_table.iterrows():
            top_entries.append(
                f"{r['cmdb_id']}|{r['kpi_name']} start={r['fault_start_timestamp_utc8']} "
                f"len={int(r['fault_length_minutes'])} type={r['anomaly_type']} sev={float(r['severity_ratio']):.3g}"
            )

        # Compose compact summary string
        summary = (
            f"Total fault segments={total_fault_segments}; "
            f"counts_by_prefix=[{', '.join(prefix_counts_list)}]; "
            f"earliest_start={earliest_fault_start_utc8}; "
            f"Top10 (severity desc): " + "; ".join(top_entries)
        )

    summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df exists
if 'trace_df' not in globals():
    raise RuntimeError("trace_df missing")

# Convert window bounds (UTC+8) to UTC epoch seconds
start_ts = int(pd.to_datetime('2020-05-30 04:00:00+0800').tz_convert('UTC').timestamp())
end_ts = int(pd.to_datetime('2020-05-30 04:30:00+0800').tz_convert('UTC').timestamp())

# Filter trace_df to the incident window (inclusive) and store in kernel variable
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 1) Unique trace cmdb_ids present in the window
unique_cmdb_ids = pd.Series(sorted(trace_window_df['cmdb_id'].dropna().unique()))

# 2) Overall min and max timestamps present in the filtered trace rows (UTC+8 strings)
if trace_window_df.shape[0] == 0:
    min_ts_utc8 = None
    max_ts_utc8 = None
else:
    min_ts = int(trace_window_df['timestamp'].min())
    max_ts = int(trace_window_df['timestamp'].max())
    min_ts_utc8 = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    max_ts_utc8 = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
time_bounds_df = pd.DataFrame([{'min_timestamp_utc8': min_ts_utc8, 'max_timestamp_utc8': max_ts_utc8}])

# 3) For each cmdb_id compute mean of trace.self.duration_mean, count, earliest timestamp in window
trace_self = trace_window_df[trace_window_df['trace_name'] == 'trace.self.duration_mean'].copy()
if trace_self.shape[0] == 0:
    per_cmdb_table = pd.DataFrame(columns=['cmdb_id','mean_duration','count','earliest_timestamp_utc8'])
else:
    grp = trace_self.groupby('cmdb_id', as_index=False)
    per_cmdb_table = grp.agg(
        mean_duration = ('value', 'mean'),
        count = ('value', 'count'),
        earliest_ts = ('timestamp', 'min')
    )
    per_cmdb_table['earliest_timestamp_utc8'] = per_cmdb_table['earliest_ts'].apply(
        lambda ts: pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    )
    per_cmdb_table = per_cmdb_table[['cmdb_id','mean_duration','count','earliest_timestamp_utc8']].sort_values('mean_duration', ascending=False).reset_index(drop=True)
    per_cmdb_table['mean_duration'] = per_cmdb_table['mean_duration'].astype(float)

# limit to top 20 rows
per_cmdb_table = per_cmdb_table.head(20)

# 4) Top 10 distinct trace_name values by overall count in the window
if trace_window_df.shape[0] == 0:
    top_trace_names = pd.DataFrame(columns=['trace_name','count'])
else:
    top_trace_names = trace_window_df.groupby('trace_name', as_index=False).size().rename(columns={'size':'count'}).sort_values('count', ascending=False).head(10).reset_index(drop=True)

# Return only the requested aggregates
unique_cmdb_ids, time_bounds_df, per_cmdb_table, top_trace_names

```
Out[8]:
```
```python
# Compose a concise plain-English summary using existing kernel variables:
# unique_cmdb_ids, time_bounds_df, per_cmdb_table, top_trace_names

uids = unique_cmdb_ids.tolist() if 'unique_cmdb_ids' in globals() else []
time_bounds = time_bounds_df.iloc[0].to_dict() if 'time_bounds_df' in globals() and not time_bounds_df.empty else {'min_timestamp_utc8': None, 'max_timestamp_utc8': None}
per_cmdb_empty = (('per_cmdb_table' in globals()) and per_cmdb_table.shape[0] == 0)
top_traces_list = []
if 'top_trace_names' in globals() and not top_trace_names.empty:
    for _, r in top_trace_names.iterrows():
        top_traces_list.append(f"{r['trace_name']} (count={int(r['count'])})")

summary = (
    "Trace window summary: "
    f"cmdb_ids_in_window={uids}; "
    f"time_range_min={time_bounds['min_timestamp_utc8']}, max={time_bounds['max_timestamp_utc8']}. "
    f"trace.self.duration_mean rows in window: {'none' if per_cmdb_empty else per_cmdb_table.shape[0]}. "
    f"Top trace_name by count: [{'; '.join(top_traces_list)}]."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
dtype: object,          min_timestamp_utc8        max_timestamp_utc8
0  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800, Empty DataFrame
Columns: [cmdb_id, mean_duration, count, earliest_timestamp_utc8]
Index: [],                        trace_name  count
0       trace.self.jdbc.row_count    240
1      trace.self.jdbc.error_rate    240
2    trace.self.jdbc.duration_p95    240
3   trace.self.jdbc.duration_mean    240
4      trace.self.local.row_count    240
5     trace.self.local.error_rate    240
6   trace.self.local.duration_p95    240
7  trace.self.local.duration_mean    240
8        trace.self.csf.row_count    180
9       trace.self.csf.error_rate    180)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Check existence of faults_df
if 'faults_df' not in globals():
    prefix_agg_df = pd.DataFrame({'message': ['faults_df missing']})
    top_components_df = pd.DataFrame({'message': ['faults_df missing']})
else:
    df = faults_df.copy()
    if df.shape[0] == 0:
        # No faults
        prefix_agg_df = pd.DataFrame(columns=[
            'prefix','fault_segment_count','earliest_fault_start_utc8',
            'latest_fault_end_utc8','max_severity_ratio','mean_severity_ratio'
        ])
        top_components_df = pd.DataFrame(columns=[
            'cmdb_id','count_of_fault_segments','earliest_start_utc8','max_severity_ratio'
        ])
    else:
        # Ensure numeric
        df['severity_ratio'] = pd.to_numeric(df['severity_ratio'], errors='coerce')

        # Parse start/end timestamps (strings include +0800) -> timezone-aware -> convert to Asia/Shanghai
        start_dt = pd.to_datetime(df['fault_start_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        end_dt = pd.to_datetime(df['fault_end_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        df['_start_dt'] = start_dt
        df['_end_dt'] = end_dt

        # Extract prefix token before first '_'
        df['prefix'] = df['cmdb_id'].astype(str).str.split('_', n=1).str[0]

        # Aggregate by prefix
        pref_grp = df.groupby('prefix', as_index=False)
        prefix_agg_df = pref_grp.agg(
            fault_segment_count = ('cmdb_id', 'size'),
            earliest_fault_start = ('_start_dt', 'min'),
            latest_fault_end = ('_end_dt', 'max'),
            max_severity_ratio = ('severity_ratio', 'max'),
            mean_severity_ratio = ('severity_ratio', 'mean')
        )

        # Format datetime columns to UTC+8 strings
        prefix_agg_df['earliest_fault_start_utc8'] = prefix_agg_df['earliest_fault_start'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        prefix_agg_df['latest_fault_end_utc8'] = prefix_agg_df['latest_fault_end'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

        # Select and order columns, sort by fault_segment_count desc for readability
        prefix_agg_df = prefix_agg_df[[
            'prefix','fault_segment_count','earliest_fault_start_utc8',
            'latest_fault_end_utc8','max_severity_ratio','mean_severity_ratio'
        ]].sort_values('fault_segment_count', ascending=False).reset_index(drop=True)

        # Aggregate by cmdb_id for top components
        comp_grp = df.groupby('cmdb_id', as_index=False)
        comps = comp_grp.agg(
            count_of_fault_segments = ('cmdb_id', 'size'),
            earliest_start = ('_start_dt', 'min'),
            max_severity_ratio = ('severity_ratio', 'max')
        )
        comps['earliest_start_utc8'] = comps['earliest_start'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        top_components_df = comps[[
            'cmdb_id','count_of_fault_segments','earliest_start_utc8','max_severity_ratio'
        ]].sort_values('max_severity_ratio', ascending=False).head(5).reset_index(drop=True)

# Return the two compact aggregates
prefix_agg_df, top_components_df

```
Out[9]:
```
# Compose a concise plain-English summary from prefix_agg_df and top_components_df
try:
    pa = prefix_agg_df
    tc = top_components_df
except NameError:
    summary = "prefix_agg_df or top_components_df missing"
else:
    if pa is None or pa.shape[0] == 0:
        summary = "No prefix aggregates available."
    else:
        total_prefixes = int(pa.shape[0])
        # build prefix summaries
        prefix_parts = []
        for _, r in pa.iterrows():
            prefix_parts.append(
                f"{r['prefix']}={int(r['fault_segment_count'])} (start={r['earliest_fault_start_utc8']}, end={r['latest_fault_end_utc8']})"
            )
        # overall earliest start (min of earliest_fault_start_utc8)
        earliest_overall = pa['earliest_fault_start_utc8'].min()
        latest_overall = pa['latest_fault_end_utc8'].max()
        # top components list
        top_parts = []
        if tc is None or tc.shape[0] == 0:
            top_msg = "no top components"
        else:
            for _, r in tc.iterrows():
                top_parts.append(
                    f"{r['cmdb_id']} (segments={int(r['count_of_fault_segments'])}, earliest={r['earliest_start_utc8']}, max_sev={float(r['max_severity_ratio']):.6g})"
                )
            top_msg = "; ".join(top_parts)
        summary = (
            f"Faults by prefix (total_prefixes={total_prefixes}): " +
            "; ".join(prefix_parts) +
            f". Overall earliest_start={earliest_overall}, latest_end={latest_overall}. Top-5 components by max severity: {top_msg}."
        )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   prefix  fault_segment_count earliest_fault_start_utc8     latest_fault_end_utc8  max_severity_ratio  mean_severity_ratio
0      os                   86  2020-05-30 04:00:00+0800  2020-05-30 04:30:00+0800        3.648102e+18         4.249457e+16
1      db                   70  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800        1.900000e+11         4.825886e+09
2   redis                   21  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800        4.070000e+12         2.414286e+11
3  docker                    4  2020-05-30 04:03:00+0800  2020-05-30 04:25:00+0800        3.783784e+00         2.748763e+00,      cmdb_id  count_of_fault_segments       earliest_start_utc8  max_severity_ratio
0     os_021                       10  2020-05-30 04:00:00+0800        3.648102e+18
1  redis_008                        4  2020-05-30 04:00:00+0800        4.070000e+12
2     os_007                        3  2020-05-30 04:00:00+0800        2.300000e+12
3     os_018                        8  2020-05-30 04:00:00+0800        1.000000e+12
4     os_016                        2  2020-05-30 04:00:00+0800        5.500000e+11)```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Check faults_df exists
if 'faults_df' not in globals():
    os_faults_df = pd.DataFrame({'message': ['faults_df missing']})
    os_faults_df
else:
    # Reuse faults_df from kernel
    df = faults_df.copy()

    # Filter to cmdb_id starting with 'os_'
    df_os = df[df['cmdb_id'].astype(str).str.startswith('os_')].copy()

    if df_os.shape[0] == 0:
        os_faults_df = pd.DataFrame(columns=[
            'cmdb_id','count_of_fault_segments','earliest_fault_start_utc8',
            'latest_fault_end_utc8','max_severity_ratio','unique_kpi_names'
        ])
        os_faults_df
    else:
        # Ensure numeric severity
        df_os['severity_ratio'] = pd.to_numeric(df_os['severity_ratio'], errors='coerce')

        # Parse start/end timestamp strings to timezone-aware datetimes in Asia/Shanghai
        df_os['_start_dt'] = pd.to_datetime(df_os['fault_start_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        df_os['_end_dt'] = pd.to_datetime(df_os['fault_end_timestamp_utc8'], utc=True).dt.tz_convert(tz)

        # Aggregate per cmdb_id
        agg = df_os.groupby('cmdb_id', as_index=False).agg(
            count_of_fault_segments = ('cmdb_id', 'size'),
            earliest_fault_start_dt = ('_start_dt', 'min'),
            latest_fault_end_dt = ('_end_dt', 'max'),
            max_severity_ratio = ('severity_ratio', 'max'),
            unique_kpis = ('kpi_name', lambda s: ','.join(list(pd.unique(s))[:10]))
        )

        # Format datetimes back to UTC+8 strings
        agg['earliest_fault_start_utc8'] = agg['earliest_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        agg['latest_fault_end_utc8'] = agg['latest_fault_end_dt'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

        # Select and order columns
        os_faults_df = agg[[
            'cmdb_id','count_of_fault_segments','earliest_fault_start_utc8',
            'latest_fault_end_utc8','max_severity_ratio','unique_kpis'
        ]].copy()

        # Sort by count desc then max_severity_ratio desc and limit to top 20 for display
        os_faults_df = os_faults_df.sort_values(['count_of_fault_segments','max_severity_ratio'], ascending=[False, False]).reset_index(drop=True)

        # Display compact result (top 20)
        os_faults_df.head(20)

```
Out[10]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

if 'os_faults_df' not in globals():
    summary = "os_faults_df missing"
else:
    df = os_faults_df.copy()
    if df.shape[0] == 0:
        summary = "No os_* fault segments detected."
    else:
        total_components = int(df.shape[0])
        # ensure sorting
        df = df.sort_values(['count_of_fault_segments','max_severity_ratio'], ascending=[False, False]).reset_index(drop=True)
        top_n = df.head(5)
        parts = []
        for _, r in top_n.iterrows():
            parts.append(
                f"{r['cmdb_id']}: count={int(r['count_of_fault_segments'])}, "
                f"earliest={r['earliest_fault_start_utc8']}, max_sev={float(r['max_severity_ratio']):.6g}, "
                f"kpis={r.get('unique_kpis','')}"
            )
        summary = (
            f"os_* components analyzed={total_components}. Top 5 by segment count/max severity: " +
            "; ".join(parts)
        )

summary

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Prepare empty outputs in case sources are missing or empty
os_021_kpi_agg_df = pd.DataFrame(columns=[
    'kpi_name','fault_segment_count','earliest_fault_start_utc8',
    'latest_fault_end_utc8','max_severity_ratio'
])
os_021_first_anomaly = pd.DataFrame(columns=[
    'timestamp_utc8','cmdb_id','kpi_name','value','p95','p5','anomaly_type'
])

# 1) Aggregate faults_df for cmdb_id == 'os_021' by kpi_name
if 'faults_df' in globals() and isinstance(faults_df, pd.DataFrame) and not faults_df.empty:
    dff = faults_df.copy()
    df_os021 = dff[dff['cmdb_id'] == 'os_021'].copy()
    if not df_os021.empty:
        # Parse start/end timestamps to timezone-aware datetimes (they include +0800)
        df_os021['_start_dt'] = pd.to_datetime(df_os021['fault_start_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        df_os021['_end_dt'] = pd.to_datetime(df_os021['fault_end_timestamp_utc8'], utc=True).dt.tz_convert(tz)
        df_os021['severity_ratio'] = pd.to_numeric(df_os021['severity_ratio'], errors='coerce')

        agg = df_os021.groupby('kpi_name', as_index=False).agg(
            fault_segment_count = ('kpi_name','size'),
            earliest_fault_start = ('_start_dt','min'),
            latest_fault_end = ('_end_dt','max'),
            max_severity_ratio = ('severity_ratio','max')
        )

        agg['earliest_fault_start_utc8'] = agg['earliest_fault_start'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
        agg['latest_fault_end_utc8'] = agg['latest_fault_end'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

        os_021_kpi_agg_df = agg[[
            'kpi_name','fault_segment_count','earliest_fault_start_utc8',
            'latest_fault_end_utc8','max_severity_ratio'
        ]].sort_values(['fault_segment_count','max_severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# 2) From incident_window_df find earliest anomaly row for cmdb_id == 'os_021'
if 'incident_window_df' in globals() and isinstance(incident_window_df, pd.DataFrame) and not incident_window_df.empty:
    diw = incident_window_df.copy()
    # ensure numeric and timestamp types
    diw['value'] = pd.to_numeric(diw['value'], errors='coerce')
    diw['p95'] = pd.to_numeric(diw.get('p95', np.nan), errors='coerce')
    diw['p5'] = pd.to_numeric(diw.get('p5', np.nan), errors='coerce')
    diw['timestamp'] = diw['timestamp'].astype(int)

    # ensure is_anomaly flag exists/recomputed
    diw['is_anomaly'] = (diw['value'] >= diw['p95']) | (diw['value'] <= diw['p5'])

    os21_anoms = diw[(diw['cmdb_id'] == 'os_021') & (diw['is_anomaly'])].copy()
    if not os21_anoms.empty:
        earliest_idx = os21_anoms['timestamp'].idxmin()
        row = os21_anoms.loc[earliest_idx]

        ts_utc8 = pd.to_datetime(int(row['timestamp']), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        # derived anomaly_type per instruction: 'high' if value >= p95 else 'low'
        try:
            anomaly_type = 'high' if float(row['value']) >= float(row['p95']) else 'low'
        except Exception:
            # fallback if p95 is NaN
            anomaly_type = 'high' if row['value'] >= row['p95'] else 'low'

        os_021_first_anomaly = pd.DataFrame([{
            'timestamp_utc8': ts_utc8,
            'cmdb_id': row['cmdb_id'],
            'kpi_name': row['kpi_name'],
            'value': float(row['value']) if pd.notna(row['value']) else np.nan,
            'p95': float(row['p95']) if pd.notna(row['p95']) else np.nan,
            'p5': float(row['p5']) if pd.notna(row['p5']) else np.nan,
            'anomaly_type': anomaly_type
        }])

# Keep results in kernel variables and provide compact outputs
os_021_kpi_agg_df, os_021_first_anomaly

```
Out[11]:
```
- Per-kpi faults for os_021 (7 KPIs):
  - CPU_iowait_time: 3 segments, earliest start 2020-05-30 04:00:00+0800, latest end 2020-05-30 04:11:00+0800, max_severity_ratio ≈ 2.365589
  - Disk_avgqu_sz: 2 segments, earliest start 2020-05-30 04:08:00+0800, latest end 2020-05-30 04:11:00+0800, max_severity_ratio ≈ 2.647591
  - Recv_total: 1 segment, earliest start 2020-05-30 04:00:00+0800, latest end 2020-05-30 04:06:00+0800, max_severity_ratio = 3.648102e+18
  - Send_total: 1 segment, earliest start 2020-05-30 04:00:00+0800, latest end 2020-05-30 04:06:00+0800, max_severity_ratio = 6.423091e+15
  - Disk_svctm: 1 segment, earliest start 2020-05-30 04:00:00+0800, latest end 2020-05-30 04:29:00+0800, max_severity_ratio = 2.0e+12
  - System_wait_queue_length: 1 segment, earliest start 2020-05-30 04:12:00+0800, latest end 2020-05-30 04:12:00+0800, max_severity_ratio = 5.5e+11
  - Disk_io_util: 1 segment, earliest start 2020-05-30 04:00:00+0800, latest end 2020-05-30 04:29:00+0800, max_severity_ratio = 2.5e+11

(These per-kpi aggregates are stored as os_021_kpi_agg_df.)

- Earliest anomaly point for os_021 (from incident_window_df):
  - timestamp_utc8: 2020-05-30 04:00:00+0800
  - cmdb_id: os_021
  - kpi_name: Agent_ping
  - value: 1.0
  - p95: 1.0
  - p5: 1.0
  - anomaly_type: high

(This single-row result is stored as os_021_first_anomaly.)

The original code execution output of IPython Kernel is also provided below for reference:

(                   kpi_name  fault_segment_count earliest_fault_start_utc8     latest_fault_end_utc8  max_severity_ratio
0           CPU_iowait_time                    3  2020-05-30 04:00:00+0800  2020-05-30 04:11:00+0800        2.365589e+00
1             Disk_avgqu_sz                    2  2020-05-30 04:08:00+0800  2020-05-30 04:11:00+0800        2.647591e+00
2                Recv_total                    1  2020-05-30 04:00:00+0800  2020-05-30 04:06:00+0800        3.648102e+18
3                Send_total                    1  2020-05-30 04:00:00+0800  2020-05-30 04:06:00+0800        6.423091e+15
4                Disk_svctm                    1  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800        2.000000e+12
5  System_wait_queue_length                    1  2020-05-30 04:12:00+0800  2020-05-30 04:12:00+0800        5.500000e+11
6              Disk_io_util                    1  2020-05-30 04:00:00+0800  2020-05-30 04:29:00+0800        2.500000e+11,              timestamp_utc8 cmdb_id    kpi_name  value  p95   p5 anomaly_type
0  2020-05-30 04:00:00+0800  os_021  Agent_ping    1.0  1.0  1.0         high)```
```